# Artificial Neural Networks — Python classroom notebook

**Introduction to Neuroscience · 24 November 2026**

Three demos parallel the MATLAB scripts. Requires NumPy and Matplotlib. Run cells in order. These are simplified synthetic teaching models, not recorded neural responses.

## Demo 0 — one artificial neuron

Change the inputs, weights and bias. This example does **not** train anything.

In [ ]:
"""Demo 0 — one artificial neuron (parallel to ANN_Demo0_OneNeuron.m).

Requires only Python and NumPy. These input values are illustrative, not neural recordings.
Run: python ANN_Demo0_OneNeuron.py
"""
import numpy as np

# Inputs: synthetic activity of left- and right-preferring populations.
x = np.array([0.8, 0.2])
w = np.array([-2.0, 2.0])  # weights chosen by hand, not learned here
b = 0.0                    # decision bias

z = float(x @ w + b)
p_right = 1.0 / (1.0 + np.exp(-z))

print(f"Weighted input z = {z:.2f}")
print(f"P(right) = {p_right:.3f}")
print("Decision:", "RIGHT" if p_right >= 0.5 else "LEFT")

# EXERCISES
# 1. Change x to [0.2, 0.8]. What happens?
# 2. Set both weights to zero. What does the neuron predict?
# 3. Change b to +2. How does bias affect the decision?
# 4. Why is this NOT yet a learning demonstration?


## Demo 1 — learning a sensory decision

Synthetic left/right-preferring inputs are generated and separated into training and held-out test trials. Change `sensory_noise` to 0.2, 0.6, or 1.2; rerun and observe the held-out accuracy.

In [ ]:
"""Demo 1 — train one artificial decision neuron on synthetic sensory activity.

Parallel to ANN_Demo1_SensoryDecision.m: same task, model, parameters,
learning rule, and plots. Python and MATLAB draw different random trials,
so individual numbers will not match exactly, even with seed 12.

Requires: NumPy, Matplotlib. Run: python ANN_Demo1_SensoryDecision.py
"""
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)
sensory_noise = 0.6        # TRY: 0.2, 0.6, 1.2
n_train, n_test = 240, 120
learning_rate = 0.4
n_epochs = 400


def motion_trials(n, noise):
    """Synthetic left/right inputs; labels 0=left, 1=right.

    This is an illustrative sensory readout, NOT recorded MT neurons.
    """
    y = (rng.random(n) > 0.5).astype(float)
    X = noise * rng.standard_normal((n, 2))
    X[:, 0] += 0.9 * (1.0 - y)  # left-preferring input
    X[:, 1] += 0.9 * y          # right-preferring input
    return X, y


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))


# Independent training and held-out test trials.
X_train, y_train = motion_trials(n_train, sensory_noise)
X_test, y_test = motion_trials(n_test, sensory_noise)

# Train a single sigmoid neuron by supervised gradient descent.
w = np.zeros(2)
b = 0.0
loss = np.zeros(n_epochs)
for epoch in range(n_epochs):
    p = sigmoid(X_train @ w + b)
    p_safe = np.clip(p, 1e-10, 1.0 - 1e-10)
    loss[epoch] = -np.mean(y_train*np.log(p_safe) + (1-y_train)*np.log(1-p_safe))

    error_signal = p - y_train
    grad_w = X_train.T @ error_signal / n_train
    grad_b = error_signal.mean()
    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

p_test = sigmoid(X_test @ w + b)
pred_test = (p_test >= 0.5).astype(float)
accuracy = np.mean(pred_test == y_test)

print("DEMO 1 - NOISY SENSORY DECISIONS")
print(f"Noise level: {sensory_noise:.2f}")
print(f"Weights: left = {w[0]:+.3f}, right = {w[1]:+.3f}; bias = {b:+.3f}")
print(f"Test accuracy: {100*accuracy:.1f}% (chance expectation = 50%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(np.arange(1, n_epochs+1), loss, linewidth=2)
ax1.set(xlabel="Training epoch", ylabel="Cross-entropy loss",
        title="Training reduces prediction error")
ax1.grid(True)

left = y_test == 0
right = y_test == 1
ax2.scatter(X_test[left, 0], X_test[left, 1], label="Left trials", s=30)
ax2.scatter(X_test[right, 0], X_test[right, 1], label="Right trials", s=30)
xs = np.linspace(X_test[:, 0].min()-.2, X_test[:, 0].max()+.2, 150)
if abs(w[1]) > 1e-9:
    ys = -(w[0]*xs + b) / w[1]
    ax2.plot(xs, ys, "k--", linewidth=2, label="Decision boundary")
ax2.set(xlabel="Left-preferring input (synthetic)",
        ylabel="Right-preferring input (synthetic)",
        title=f"Unseen trials: {100*accuracy:.1f}% correct")
ax2.set_aspect("equal", adjustable="box")
ax2.legend()
ax2.grid(True)
fig.tight_layout()
plt.show()

# QUESTIONS
# 1. Does test accuracy decrease as sensory_noise increases?
# 2. Does changing noise change the input observations, the learning rule, or both?
# 3. What do the signs of the learned weights mean?
# 4. Why is performance evaluated on new trials?
# IMPORTANT: supervised gradient descent is NOT Hebbian LTP.


## Demo 2 — the XOR problem

A linear neuron cannot separate the XOR cases. A network with a nonlinear hidden layer can. This is a demonstration on the **same four training cases**; it is not a generalization experiment.

In [ ]:
"""Demo 2 — linear neuron versus one nonlinear hidden layer on XOR.

Parallel to ANN_Demo2_XOR.m: the same four examples, exact starting
weights, 4 tanh hidden units, sigmoid output, gradient descent, and plots.
The four examples are BOTH training and evaluation: this is NOT a test
of generalization to new data.

Requires: NumPy, Matplotlib. Run: python ANN_Demo2_XOR.py
"""
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
t = np.array([0., 1., 1., 0.])
n = len(t)
eta = 1.0
n_epochs = 7000


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))


# First: one linear sigmoid neuron cannot separate XOR classes.
w_linear = np.zeros(2)
b_linear = 0.0
for epoch in range(n_epochs):
    p = sigmoid(X @ w_linear + b_linear)
    delta = (p - t)/n
    w_linear -= eta * (X.T @ delta)
    b_linear -= eta * delta.sum()
p_linear = sigmoid(X @ w_linear + b_linear)

# Next: 2 inputs -> 4 tanh hidden units -> 1 sigmoid output.
# Same fixed initial weights as the MATLAB version.
W1 = np.array([[0.6, -0.9, 0.5, 1.2],
               [0.8,  0.7, -1.1, -0.5]])
b1 = np.zeros(4)
W2 = np.array([0.3, -0.5, 0.8, -0.2])
b2 = 0.0
loss = np.zeros(n_epochs)

for epoch in range(n_epochs):
    H = np.tanh(X @ W1 + b1)
    p = sigmoid(H @ W2 + b2)
    p_safe = np.clip(p, 1e-10, 1-1e-10)
    loss[epoch] = -np.mean(t*np.log(p_safe) + (1-t)*np.log(1-p_safe))

    # Backpropagation in NumPy: propagate prediction error backward.
    delta_out = (p-t)/n
    grad_W2 = H.T @ delta_out
    grad_b2 = delta_out.sum()
    delta_hidden = np.outer(delta_out, W2)*(1-H**2)
    grad_W1 = X.T @ delta_hidden
    grad_b1 = delta_hidden.sum(axis=0)

    W1 -= eta*grad_W1
    b1 -= eta*grad_b1
    W2 -= eta*grad_W2
    b2 -= eta*grad_b2

H = np.tanh(X @ W1 + b1)
p_network = sigmoid(H @ W2 + b2)

print("DEMO 2 - XOR")
print("x1 x2 | target | one neuron | hidden layer")
for k in range(n):
    print(f" {int(X[k,0])}  {int(X[k,1])} |   {int(t[k])}    |   {p_linear[k]:.3f}    |    {p_network[k]:.3f}")
print(f"Linear training accuracy: {100*np.mean((p_linear>=0.5)==t):.0f}%")
print(f"Network training accuracy: {100*np.mean((p_network>=0.5)==t):.0f}%")
print("NOTE: these are the 4 training cases, not an independent test set.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(np.arange(1, n_epochs+1), loss, linewidth=2)
ax1.set(xlabel="Training epoch", ylabel="Cross-entropy loss",
        title="Hidden-layer network learns XOR")
ax1.grid(True)

xx, yy = np.meshgrid(np.linspace(-.25, 1.25, 100),
                     np.linspace(-.25, 1.25, 100))
Q = np.column_stack((xx.ravel(), yy.ravel()))
HQ = np.tanh(Q @ W1 + b1)
pq = sigmoid(HQ @ W2 + b2).reshape(xx.shape)
ax2.contour(xx, yy, pq, levels=[0.5], linewidths=2)
ax2.scatter(X[t==0,0], X[t==0,1], marker="o", label="Target 0", s=100)
ax2.scatter(X[t==1,0], X[t==1,1], marker="s", label="Target 1", s=100)
ax2.set(xlabel="Input x1", ylabel="Input x2", title="Nonlinear decision boundary")
ax2.set(xlim=(-.3, 1.3), ylim=(-.3, 1.3))
ax2.set_aspect("equal", adjustable="box")
ax2.grid(True)
ax2.legend()
fig.tight_layout()
plt.show()

# QUESTIONS
# 1. Can one straight line separate the XOR classes?
# 2. Why does adding nonlinear hidden units change what the network can learn?
# 3. Is 100% accuracy on four training cases evidence of generalization? (No.)
# 4. Try reducing the hidden-unit count. Update W1, b1, W2 consistently.
# CAUTION: artificial hidden layers are not anatomical cortical layers.


## Neuroscience discussion

1. Which biological ideas motivate the concepts of units and connection weights?
2. How does supervised error correction differ from Hebbian LTP/LTD?
3. How does a learned readout relate conceptually to sensory encoding in MT and decision processing in LIP?
4. Why can an ANN hidden layer not be automatically identified with a cortical layer?